In [12]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
df=pd.read_csv('question_ans_analysis.csv')

In [ ]:
from ydata_profiling import ProfileReport
profile = ProfileReport(df, title="Data Profile")
profile.to_notebook_iframe()

In [ ]:
print("First 5 rows:")
print(df.head())

First 5 rows:
                                       question_text           subject  \
0    Solve the quadratic equation scenario number 1.       Mathematics   
1     Implement binary search for scenario number 2.  Computer Science   
2      Apply Newton law to system scenario number 3.           Physics   
3  Evaluate matrix determinant for scenario numbe...       Mathematics   
4    Solve the quadratic equation scenario number 5.       Mathematics   

  cognitive_level_bloom  readability_score  word_count  sentence_count  \
0                create              77.49          22               1   
1            understand              45.01          17               2   
2                create              89.84          36               1   
3              remember              44.38          17               1   
4            understand              54.48          23               2   

   time_taken_minutes  total_students_attempted  correct_attempts  \
0                  23      

In [ ]:
print("\nInfo:")
df.info()



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   question_text             5000 non-null   object 
 1   subject                   5000 non-null   object 
 2   cognitive_level_bloom     5000 non-null   object 
 3   readability_score         5000 non-null   float64
 4   word_count                5000 non-null   int64  
 5   sentence_count            5000 non-null   int64  
 6   time_taken_minutes        5000 non-null   int64  
 7   total_students_attempted  5000 non-null   int64  
 8   correct_attempts          5000 non-null   int64  
 9   incorrect_attempts        5000 non-null   int64  
 10  correct_percentage        5000 non-null   float64
 11  learning_gap_score        5000 non-null   float64
 12  discrimination_index      5000 non-null   float64
 13  difficulty_label          5000 non-null   object 
 14  a

In [ ]:
print("\nMissing values:")
print(df.isnull().sum())


Missing values:
question_text               0
subject                     0
cognitive_level_bloom       0
readability_score           0
word_count                  0
sentence_count              0
time_taken_minutes          0
total_students_attempted    0
correct_attempts            0
incorrect_attempts          0
correct_percentage          0
learning_gap_score          0
discrimination_index        0
difficulty_label            0
assessment_quality_score    0
dtype: int64


In [ ]:
columns_to_drop = [
    "correct_percentage",
    "learning_gap_score",
    "discrimination_index",
    "assessment_quality_score"
]

df = df.drop(columns=columns_to_drop, errors="ignore")

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_question"] = df["question_text"].apply(clean_text)

In [ ]:
X_text = df["clean_question"]

In [ ]:
numeric_features = [
    "readability_score",
    "word_count",
    "sentence_count",
    "total_students_attempted",
    "correct_attempts",
    "incorrect_attempts"
]

X_numeric = df[numeric_features]

In [ ]:
y = df["difficulty_label"]

In [ ]:
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text,
    X_numeric,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words="english"
)

X_text_train_tfidf = vectorizer.fit_transform(X_text_train)
X_text_test_tfidf = vectorizer.transform(X_text_test)

In [ ]:
scaler = StandardScaler()

X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled = scaler.transform(X_num_test)

In [ ]:
X_train_final = np.hstack((X_text_train_tfidf.toarray(), X_num_train_scaled))
X_test_final = np.hstack((X_text_test_tfidf.toarray(), X_num_test_scaled))

In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    C=1.0
)

lr_model.fit(X_train_final, y_train)

y_pred_lr = lr_model.predict(X_test_final)

print("===== Logistic Regression =====")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lr))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_lr))

===== Logistic Regression =====
Accuracy: 0.99

Classification Report:

              precision    recall  f1-score   support

        easy       1.00      0.99      0.99       496
        hard       1.00      0.99      0.99       511
      medium       0.97      1.00      0.98       493

    accuracy                           0.99      1500
   macro avg       0.99      0.99      0.99      1500
weighted avg       0.99      0.99      0.99      1500


Confusion Matrix:

[[489   0   7]
 [  0 505   6]
 [  1   1 491]]


In [ ]:
dt_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

dt_model.fit(X_train_final, y_train)

y_pred_dt = dt_model.predict(X_test_final)

print("\n===== Decision Tree =====")
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_dt))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_dt))


===== Decision Tree =====
Accuracy: 0.9826666666666667

Classification Report:

              precision    recall  f1-score   support

        easy       0.99      0.99      0.99       496
        hard       0.99      0.97      0.98       511
      medium       0.96      0.98      0.97       493

    accuracy                           0.98      1500
   macro avg       0.98      0.98      0.98      1500
weighted avg       0.98      0.98      0.98      1500


Confusion Matrix:

[[491   0   5]
 [  0 498  13]
 [  5   3 485]]


In [ ]:
cv_scores = cross_val_score(
    lr_model,
    X_train_final,
    y_train,
    cv=5
)

print("\nCross Validation Accuracy:", cv_scores.mean())


Cross Validation Accuracy: 0.9888571428571428


In [ ]:
joblib.dump(lr_model, "lr_model.pkl")
joblib.dump(vectorizer, "tfidf.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']